In [ ]:
from option_chain_downloader import OptionChainDownloader
from option_finder import *
from option_data_plotter import *
import matplotlib.pyplot as plt

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/all_options.log')

In [ ]:
chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
ocd = OptionChainDownloader(chain_dir, quotes_dir, cookie_file, logger, strikes='ALL')
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)
print('OptionFinder chain_dir:', self.chain_dir)

In [ ]:
symlist  = ['QQQ', 'GLD', 'TLT', 'IBIT']
symlist += ['SPY', 'DIA', 'XLF', 'CRCL']
symlist += ['ETHA', 'NVDA', 'AAPL', 'GOOL']
symlist += ['MSFT', 'TSLA', 'PLTR', 'META']
symlist += ['AVGO', 'AMZN', 'TSM', 'SMH']

## Refresh data here

In [ ]:
_t0 = time.time()
_n = ocd.download_option_chain(symlist, batch_size=5, rps=5)
print(_n, 'files downloaded in', int(time.time() - _t0), 'seconds')

In [ ]:
self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=800, height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(30).tail(20).T

### 0 DTE PUT

In [ ]:
df0d = dfcp[(dfcp.dte==0) & (dfcp.type=='P')].drop(columns=['cluster', 'dte_cluster', 'dte', 'expDt', 'type'])
df0d = df0d[(df0d.Bid >= 0.5) & (df0d.OpenInterest >= 5) & (df0d.moneyness < 1.0) & (df0d.pctSpread <= 10)]
df0d.sort_values(by='mid', ascending=False).head(60)

## Total open interests and volumes for all dte and strikes

In [ ]:
_df = dfcp.loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False)

### It may be of interest to look at OpenInterests and Volumes for 8-weeks and 1-year DTE clusters

In [ ]:
for _dte_cluster in ['8wk', '1yr']:
    _df = dfcp[dfcp.dte_cluster==_dte_cluster].loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
    plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False, log_y_threshold=500, horizontal_spacing=0.03)

### Ratios of volume/openinterest

In [ ]:
dfvo = aggregate_metrics_by_moneyness_and_dte_clusters(dfcp, ['Volume', 'OpenInterest'], method='sum', oi_lb=None, bid_lb=0)
dfvo['Volume_OpenInterest_ratio'] = dfvo.Volume/dfvo.OpenInterest
dfvo = dfvo.drop(columns=['Volume', 'OpenInterest'])

In [ ]:
_dte_cluster = '1wk'
for _type in ['C', 'P']:
    _df = dfvo[(dfvo.dte_cluster==_dte_cluster) & (dfvo.type==_type)].drop(columns=['dte_cluster', 'type'])
    _metric = _df.columns[-1]
    _metric2 = f'{_type} option {_dte_cluster} {_metric}'
    plot_metrics_in_one_row(_df.rename(columns={_metric: _metric2}), ['symbol', 'cluster'], [_metric2], shared_y=True, log_y_threshold=500, horizontal_spacing=0.02)

In [ ]:
plot_metrics_by_moneyness_and_dte_clusters(dfvo[(dfvo.type=='C') & (dfvo.dte_cluster != '1wk')], height=240, vertical_spacing=0.03, shared_yaxes=True)

In [ ]:
_df = calc_overall_put_call_ratios(dfcp)
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_df = calc_overall_put_call_ratios(dfcp[dfcp.cluster=='atm'])
_df = _df.rename(columns=dict([(c, 'atm '+c) for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_df = calc_overall_put_call_ratios(dfcp[(dfcp.cluster=='atm') & (dfcp.dte >= 90) & (dfcp.dte <= 180)])
_df = _df.rename(columns=dict([(c, 'atm dte 90 to 180 '+c) for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:], shared_y=False)

In [ ]:
dte_filter = ~ dfcp.dte_cluster.str.contains('^(?:1w|>1y)')
_df = calc_cluster_put_call_ratios(dfcp[dte_filter])#[~ (dfcp.cluster.str.contains('deep')
plot_metrics_by_moneyness_and_dte_clusters(_df, height=230, vertical_spacing=0.03, shared_yaxes=False)

Which metrics can be clustered by moneyness/dte? How to compare options between symbols?
pctProfit, pctSpread, theta, overpaid, leverage

### Average pctSpread of put options by moneyness and DTE clusters

In [ ]:
_df = aggregate_metrics_by_moneyness_and_dte_clusters(dfcp, 'pctSpread', method='mean', oi_lb=100, bid_lb=0)
_filter = (_df.type == 'P') & (~ _df.dte_cluster.str.contains(r'^(?:1wk|>1yr)')) & (~ _df.cluster.str.contains(r'^(?:deep)'))
plot_metrics_by_moneyness_and_dte_clusters(_df[_filter], height=230, vertical_spacing=0.03, shared_yaxes=False)

In [ ]:
_symbol = 'QQQ'
_df = dfcp[dfcp.symbol==_symbol]
oi_lb = _df[_df.OpenInterest>0].OpenInterest.quantile(0.1)
print('OpenInterest lb:', oi_lb, 'Spot price:', _df.lastPrice.values[0])
px.scatter(_df[(_df.OpenInterest >= oi_lb) & (_df.Bid >= 10)], x='OpenInterest', y='pctSpread', color='type')

In [ ]:
_dte = 35
_metric = 'Theta'#'OpenInterest'
_df = dfcp[(dfcp.symbol==_symbol) & (dfcp.dte==_dte)]
px.scatter(_df[(_df.strike >= 500) & (_df.strike <= 600)], x='strike', y=_metric, color='type', title=f'{_symbol} {_metric} dte: {_dte}')

In [ ]:
_p = _df[_df.strike <= 625].pivot(columns=['type'], index='strike', values=['mid'])
_p['p_c'] = _p.mid.P / _p.mid.C
_p = _p.reset_index()
_p.columns = ['strike', 'mid_C', 'mid_P', 'p_c']
#_p.reset_index().columns
px.scatter(_p, x='strike', y='p_c', title=f'{_symbol} put/call premium ratio')

In [ ]:
_symbol = 'QQQ'
_ = plot_put_call_ratios_by_moneyness(dfcp[(dfcp.symbol==_symbol) & (dfcp.dte >= 90)])

In [ ]:
_df = plot_option_details_by(dfcp, 'SPY', 'Theta', 'dte', offset=10, delta_lb=0.25, delta_ub=0.85, nr=4, nc=2)

In [ ]:
def plot_leverage_overpaid(df, delta_lb=0.5, overpaid_ub=0.1, price_lb=5, spread_ub=10, leverage_lb=2, openinterest_lb=10):
    _filter = (df.Delta >= delta_lb) & (df.overpaid <= overpaid_ub) & (df.mid >= price_lb) & (df.pctSpread <= spread_ub)
    _filter = _filter & (df.leverage >= leverage_lb) & (df.OpenInterest >= openinterest_lb)
    _df = df[_filter].set_index(['symbol', 'expDt']).sort_values(by='leverage', ascending=False)
    _symlist = list(_df.index.get_level_values('symbol').unique())
    nr = 1 if len(_symlist) <= 4 else int(np.ceil(len(_symlist)/4))
    nc = int(np.ceil(len(_symlist)/nr))
    #plt.rcParams['figure.figsize'] = (nc*5, nr*5)
    #fig, ax = plt.subplots(nr, nc)
    titles=[f'{symbol} {"~".join((lambda x: [str(x[0]), str(x[-1])])(sorted(_df.loc[symbol].dte.unique())))} DTE' for symbol in _symlist]
    fig = make_subplots(rows=nr, cols=nc, subplot_titles=titles, horizontal_spacing=0.01, vertical_spacing=0.05, shared_xaxes=True, shared_yaxes=True)
    fig.update_layout(height=nr*300)
    for i, symbol in enumerate(_symlist):
        _dte = sorted(_df.loc[symbol].dte.unique())
        print(symbol, len(_dte), 'DTEs', _dte)
        chart = px.scatter(_df.xs(symbol, level='symbol'), x='overpaid', y='leverage')
        for trace in chart.data:
            fig.add_trace(trace, row=i//nc+1, col=i%nc+1)
    fig.show()
    return _df

In [ ]:
plot_leverage_overpaid(dfcp[(dfcp.dte >= 90) & (dfcp.dte <= 360)], delta_lb=0.5, overpaid_ub=0.05, price_lb=5, spread_ub=5, leverage_lb=2, openinterest_lb=100)

### The End